# Homework 2 - Spark SQL

In this homework, you will use Spark SQL to explore, prepare, and analyze two relatively large datasets. 

- Part A: 346MB, half-million-row Yelp dataset: data exploration, splitting, and joining
- Part B: 1.2GB / 22 million-row Wikipedia clickstream dataset: sampling, analysis, aggregation

## IMPORTANT: READ THESE GENERAL REQUIREMENTS FIRST!

Throughout this homework, your code should be **idempotent**, meaning it can be applied multiple times without changing the result beyond the initial application.

- Your code should be rerun-safe (e.g., it should properly handle existing and non-existing files/tables without generating errors).
- To the extent possible, your code should generate the same result each time (e.g., keep the outcome deterministic in the case of sampling).

> We ask you to practice idempotent coding because it is often part of data/ML engineering expectations, which are important for replication, debugging, auditing, and automation.

Unless specifically noted, you are expected to **complete the tasks using code** (Spark SQL or DataFrame API code, rather than UI), with **results displayed in the notebook**. 

- For example, when asked to display the content of a `Unity catalog` folder or a file, use `dbutils.fs` functions.
- When asked to create a database, table, volume, etc., perform these actions using Spark SQL or Spark DataFrame APIs.

> This is because:
> 
> - Automation requires every task to be completed via code.
> - Code-based operations are more precise and not susceptible to UI variations and changes.
> - Code-based operations with documented outcomes in notebooks are easier to verify (for grading or auditing purposes); otherwise, you would need to describe what you have done via UI and take screenshots.

## Part A. Analyze and Prepare Yelp Academic Data Set

Yelp has held several [open data challenges](https://www.yelp.com/dataset/challenge) in the past. In this version, it assembled a database of businesses near several selected US universities, along with reviews of these businesses and reviewers.

The dataset is available as a single JSON (JSONL) file, which can be downloaded here:

- [Yelp Academic Challenge Dataset](https://idsdl.csom.umn.edu/c/share/yelp.zip): 128MB compressed, 346MB uncompressed.

### 1. Create a volume `homework2`, under the catalog `msbabigdata` and schema `spark`. 

This is where you upload the data files.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.homework2;
SHOW VOLUMES IN msbabigdata.spark;

database | volume_name
spark | homework2
spark | lab8_2

### 2. Download the zip file, unzip it, and upload it to the Unity Catalog volume `homework2`.

This step is manual, and there is NO need to report anything (process or results).

In [ ]:
yelpdf = spark.read.json("/Volumes/msbabigdata/spark/homework2/yelp.json")

# First 10 rows
yelpdf.limit(10).display()

# Column names / schema
yelpdf.printSchema()

# Row count
print(yelpdf.count())

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
3.2258064516129 | None | None | None | None | None | None | None | LiiLii C. | None | None | None | 31 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=JkeCKyEaQlbLd9uZYl4DjA | JkeCKyEaQlbLd9uZYl4DjA | [18, 15, 48]
3.0 | None | None | None | None | None | None | None | Jan Ellen T. | None | None | None | 4 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=cs91PAsv6esdWAaSkzm2lg | cs91PAsv6esdWAaSkzm2lg | [0, 0, 2]
3.60606060606061 | None | None | None | None | None | None | None | Saki U. | None | None | None | 33 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=cMgGj2FXHEbzdNZdLN_EwA | cMgGj2FXHEbzdNZdLN_EwA | [10, 8, 19]
3.3030303030303 | None | None | None | 

### 3. Create a Spark DataFrame `yelpdf` from the uploaded `yelp.json`

Then:
- Display the first 10 rows. This operation may not be fully deterministic, but some variation is acceptable for this step.
- Display the column names. You may use `.printSchema()`, `.columns`, or Spark SQL.
- Count the number of rows.

In [ ]:
yelpdf = spark.read.json("/Volumes/msbabigdata/spark/homework2/yelp.json")

# Display first 10 rows
yelpdf.limit(10).display()

# Display Column names / schema
yelpdf.printSchema()

# For Row count
print(yelpdf.count())

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
3.2258064516129 | None | None | None | None | None | None | None | LiiLii C. | None | None | None | 31 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=JkeCKyEaQlbLd9uZYl4DjA | JkeCKyEaQlbLd9uZYl4DjA | [18, 15, 48]
3.0 | None | None | None | None | None | None | None | Jan Ellen T. | None | None | None | 4 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=cs91PAsv6esdWAaSkzm2lg | cs91PAsv6esdWAaSkzm2lg | [0, 0, 2]
3.60606060606061 | None | None | None | None | None | None | None | Saki U. | None | None | None | 33 | None | None | None | None | None | user | http://www.yelp.com/user_details?userid=cMgGj2FXHEbzdNZdLN_EwA | cMgGj2FXHEbzdNZdLN_EwA | [10, 8, 19]
3.3030303030303 | None | None | None | 

### 4. Create a database called `yelp`, then save `yelpdf` as a table in the database named `yelp_raw_bronze`.

- Next, display the first 20 rows from the table.

In [ ]:
# Create database (rerun-safe)
spark.sql("CREATE DATABASE IF NOT EXISTS yelp")

# Save DataFrame as table yelp.yelp_raw_bronze
# overwrite makes it idempotent (safe to rerun)
yelpdf.write.mode("overwrite").saveAsTable("yelp.yelp_raw_bronze")

# Display first 20 rows from the table
spark.sql("SELECT * FROM yelp.yelp_raw_bronze LIMIT 20").display()

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
None | yx4IYpg9MGbrVqNdZwXX_A | None | None | 2010-04-27 | None | None | None | None | None | None | None | None | iFuFleA5MEB1FRaVej9o0Q | None | 3.0 | None | I have uncovered how to receive good service & great food @ the tft... COME!!! on week-DAY around 1ish. 

On a recent late night journey, i had great service but the food was waaack...think of mcdonalds fries left under the heat lamp all day & THEN getting the scraps on the bottom [except you're paying almost twice as much].

@ 1p on a Thursday life was easy breezy beautiful cover girl. If you have a job/school/early afternoon obligations....well, then you're screwed. try take out. | review | None | udkFd6GtlCIF92adMaFQsw | [0, 0, 0]
None | FBRxoQSWtziiU_A5tscVGA | None | None | 2008-11-27 | None | None | N

### 5. Count the number of rows by `type`

This dataset is a conglomerate of three different datasets. Its `type` field indicates the dataset type.

- Show the list of distinct `type`s and the count of rows for each type.

In [ ]:
spark.sql("""
    SELECT 
        type,
        COUNT(*) AS row_count
    FROM yelp.yelp_raw_bronze
    GROUP BY type
    ORDER BY row_count DESC
""").display()

type | row_count
review | 330071
user | 130873
business | 13490

### 6. Split the table into three tables: `reviews`, `users`, and `businesses`

You will notice there are three types of rows. For each type of row:
- Display 10 rows.
- Note which fields are used for this type of data.
- Create a new table in the `yelp` database, using only the fields relevant to the type.
- Verify the resulting table by **displaying 10 rows** and **counting the number of rows**.

> If you prefer SQL for the table creation step, you may use `CREATE OR REPLACE TABLE AS ...`.

In [ ]:
# Review rows
spark.sql("""
SELECT *
FROM yelp.yelp_raw_bronze
WHERE type = 'review'
LIMIT 10
""").display()

# User rows
spark.sql("""
SELECT *b
FROM yelp.yelp_raw_bronze
WHERE type = 'user'
LIMIT 10
""").display()

# Business rows
spark.sql("""
SELECT *
FROM yelp.yelp_raw_bronze
WHERE type = 'business'
LIMIT 10
""").display()

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
None | HSvxkRy8GQTZX56OFB4L-g | None | None | 2011-10-04 | None | None | None | None | None | None | None | None | dnmBKIAOlUiZlT4OhaeJtA | None | 3.0 | None | I've heard such mixed reviews of this place, but I honestly really like it!  I'm a creature of habit, so I always get shouyu with medium noodles.  The noodles and brother are delicious, plus the fact that they give you a whole chunk of yummy pork up in there - what's not to like?

The only reason I gave it only 3 stars is because it is a bit pricey for the amount you get, but they do have student discounts (like a "happy hour") during the day, which is nice. | review | None | tLVvCnDhu87rSXH0M8wnbg | [0, 0, 0]
None | piXuRfZ81xFGA64WFJrKkQ | None | None | 2010-08-05 | None | None | None | None | None | None

In [ ]:
%sql
-- Reviews table (only columns that exist for review rows)
CREATE OR REPLACE TABLE yelp.reviews AS
SELECT * FROM yelp.yelp_raw_bronze
WHERE type = 'review';

-- Users table
CREATE OR REPLACE TABLE yelp.users AS
SELECT * FROM yelp.yelp_raw_bronze
WHERE type = 'user';

-- Businesses table
CREATE OR REPLACE TABLE yelp.businesses AS
SELECT * FROM yelp.yelp_raw_bronze
WHERE type = 'business';

num_affected_rows | num_inserted_rows

In [ ]:
# Verify reviews
spark.sql("SELECT * FROM yelp.reviews LIMIT 10").display()
print("Reviews count:", spark.sql("SELECT COUNT(*) FROM yelp.reviews").collect()[0][0])

# Verify users
spark.sql("SELECT * FROM yelp.users LIMIT 10").display()
print("Users count:", spark.sql("SELECT COUNT(*) FROM yelp.users").collect()[0][0])

# Verify businesses
spark.sql("SELECT * FROM yelp.businesses LIMIT 10").display()
print("Businesses count:", spark.sql("SELECT COUNT(*) FROM yelp.businesses").collect()[0][0])

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
None | wbpbaWBfU54JbjLIDwERQA | None | None | 2009-06-09 | None | None | None | None | None | None | None | None | 0xuZfa0t4MNWd3eIFF02ug | None | 5.0 | None | I'm a fan of soft serve ice cream and Guptill's Coney Express has delicious ice cream with many flavors.  I've tried Kurver Kreme in Colonie, Tastee Freeze in Delmar and Country Drive Inn in Clifton Park, but I think that this place has the best soft serve ice cream.  The portions are generous and the taste is very rich.  For example, the brownie sundae is decadently delicious but likely too much for one person.  They also have cupcake sundaes which I am looking to try soon! | review | None | kT43SxDgMGzbeXpO51f0hQ | [1, 0, 0]
None | 4iTRjN_uAdAb7_YZDVHJdg | None | None | 2011-02-17 | None | None | None | N

### 7. JOIN `reviews`, `users`, and `businesses` tables

You notice that the `reviews` table has `user_id` and `business_id`, but not other user or business attributes.

You want to pre-join the three tables to make all fields available to analysts who want to do exploratory or predictive analysis.
- Make sure to include all information available in the original dataset; don't throw anything away.
- If two tables share a common field with identical values (e.g., ID fields), keep only one copy of the field.
- Include a review even if there is no corresponding reviewer (`user_id`) or business (`business_id`) in the `users` or `businesses` tables.
- If two tables have fields with the same name but store different information (e.g., both `users` and `businesses` tables have a `name` column, but they store user name and business name respectively), give them different names to denote their content (e.g., `user_name`, `business_name`).
- Save the results of the join in a new table called `reviews_silver` in the `yelp` database.
- Verify the resulting table by displaying 10 rows, the number of rows, and column names.

In [ ]:
print("REVIEWS columns:", spark.table("yelp.reviews").columns)
print("USERS columns:", spark.table("yelp.users").columns)
print("BUSINESSES columns:", spark.table("yelp.businesses").columns)

REVIEWS columns: ['average_stars', 'business_id', 'categories', 'city', 'date', 'full_address', 'latitude', 'longitude', 'name', 'neighborhoods', 'open', 'photo_url', 'review_count', 'review_id', 'schools', 'stars', 'state', 'text', 'type', 'url', 'user_id', 'votes']
USERS columns: ['average_stars', 'business_id', 'categories', 'city', 'date', 'full_address', 'latitude', 'longitude', 'name', 'neighborhoods', 'open', 'photo_url', 'review_count', 'review_id', 'schools', 'stars', 'state', 'text', 'type', 'url', 'user_id', 'votes']
BUSINESSES columns: ['average_stars', 'business_id', 'categories', 'city', 'date', 'full_address', 'latitude', 'longitude', 'name', 'neighborhoods', 'open', 'photo_url', 'review_count', 'review_id', 'schools', 'stars', 'state', 'text', 'type', 'url', 'user_id', 'votes']


In [ ]:
from pyspark.sql.functions import col, count, when
raw = spark.table("yelp.yelp_raw_bronze")
print("=== REVIEW non-null counts ===")
reviews_raw = raw.filter(col("type") == "review")
reviews_raw.select([count(when(col(c).isNotNull(), c)).alias(c) for c in raw.columns]).display()

=== REVIEW non-null counts ===

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
0 | 330071 | 0 | 0 | 330071 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 330071 | 0 | 330071 | 0 | 330071 | 330071 | 0 | 330071 | 330071

In [ ]:
print("=== USER non-null counts ===")
users_raw = raw.filter(col("type") == "user")
users_raw.select([count(when(col(c).isNotNull(), c)).alias(c) for c in raw.columns]).display()

=== USER non-null counts ===

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
130873 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 130873 | 0 | 0 | 0 | 130873 | 0 | 0 | 0 | 0 | 0 | 130873 | 130873 | 130873 | 130873

In [ ]:
print("=== BUSINESS non-null counts ===")
biz_raw = raw.filter(col("type") == "business")
biz_raw.select([count(when(col(c).isNotNull(), c)).alias(c) for c in raw.columns]).display()

=== BUSINESS non-null counts ===

average_stars | business_id | categories | city | date | full_address | latitude | longitude | name | neighborhoods | open | photo_url | review_count | review_id | schools | stars | state | text | type | url | user_id | votes
0 | 13490 | 13490 | 13490 | 0 | 13490 | 13490 | 13490 | 13490 | 13490 | 13490 | 13490 | 13490 | 0 | 13490 | 13490 | 13490 | 0 | 13490 | 13490 | 0 | 0

In [ ]:
# Recreate reviews
spark.sql("""
CREATE OR REPLACE TABLE yelp.reviews AS
SELECT review_id, user_id, business_id, stars, date, text, votes, type
FROM yelp.yelp_raw_bronze
WHERE type = 'review'
""")

spark.sql("SELECT * FROM yelp.reviews LIMIT 10").display()
print("Reviews count:", spark.sql("SELECT COUNT(*) FROM yelp.reviews").collect()[0][0])

review_id | user_id | business_id | stars | date | text | votes | type
0xuZfa0t4MNWd3eIFF02ug | kT43SxDgMGzbeXpO51f0hQ | wbpbaWBfU54JbjLIDwERQA | 5.0 | 2009-06-09 | I'm a fan of soft serve ice cream and Guptill's Coney Express has delicious ice cream with many flavors.  I've tried Kurver Kreme in Colonie, Tastee Freeze in Delmar and Country Drive Inn in Clifton Park, but I think that this place has the best soft serve ice cream.  The portions are generous and the taste is very rich.  For example, the brownie sundae is decadently delicious but likely too much for one person.  They also have cupcake sundaes which I am looking to try soon! | [1, 0, 0] | review
bBqVqhOvNgFs8I1Wk68QUQ | T9hGHsbJW9Hw1cJAlIAWmw | 4iTRjN_uAdAb7_YZDVHJdg | 5.0 | 2011-02-17 | The nurses here were very  attentive and wonderful. I was able to have the same surgical nurse that I had for another procedure a few years ago. The anesthesiologist listened to my concerns and acting skillfully and compassionately. I felt 

In [ ]:
# Recreate users
spark.sql("""
CREATE OR REPLACE TABLE yelp.users AS
SELECT user_id, name, average_stars, review_count, url, votes, type
FROM yelp.yelp_raw_bronze
WHERE type = 'user'
""")

spark.sql("SELECT * FROM yelp.users LIMIT 10").display()
print("Users count:", spark.sql("SELECT COUNT(*) FROM yelp.users").collect()[0][0])

user_id | name | average_stars | review_count | url | votes | type
JkeCKyEaQlbLd9uZYl4DjA | LiiLii C. | 3.2258064516129 | 31 | http://www.yelp.com/user_details?userid=JkeCKyEaQlbLd9uZYl4DjA | [18, 15, 48] | user
cs91PAsv6esdWAaSkzm2lg | Jan Ellen T. | 3.0 | 4 | http://www.yelp.com/user_details?userid=cs91PAsv6esdWAaSkzm2lg | [0, 0, 2] | user
cMgGj2FXHEbzdNZdLN_EwA | Saki U. | 3.60606060606061 | 33 | http://www.yelp.com/user_details?userid=cMgGj2FXHEbzdNZdLN_EwA | [10, 8, 19] | user
KXJbnHT4PDS1JZNCFKdmMg | stephanie h. | 3.3030303030303 | 33 | http://www.yelp.com/user_details?userid=KXJbnHT4PDS1JZNCFKdmMg | [78, 61, 119] | user
Tpmvufw1eea1DrjLAY2jLg | Theodore J. | 4.03846153846154 | 78 | http://www.yelp.com/user_details?userid=Tpmvufw1eea1DrjLAY2jLg | [30, 45, 74] | user
L22W35q3Ci3TytpA2LW34g | Doug H. | 4.0 | 1 | http://www.yelp.com/user_details?userid=L22W35q3Ci3TytpA2LW34g | [0, 0, 0] | user
xiJ6e1qN3Sqv4dS4D8CpNg | Amelia M. | 4.18181818181818 | 12 | http://www.yelp.com/user_det

In [ ]:
# Recreate businesses
spark.sql("""
CREATE OR REPLACE TABLE yelp.businesses AS
SELECT business_id, name, city, state, full_address, latitude, longitude,
       stars, review_count, open, categories, neighborhoods, schools,
       photo_url, url, type
FROM yelp.yelp_raw_bronze
WHERE type = 'business'
""")

spark.sql("SELECT * FROM yelp.businesses LIMIT 10").display()
print("Businesses count:", spark.sql("SELECT COUNT(*) FROM yelp.businesses").collect()[0][0])

business_id | name | city | state | full_address | latitude | longitude | stars | review_count | open | categories | neighborhoods | schools | photo_url | url | type
Iu-oeVzv8ZgP18NIB0UMqg | Southern California Medical Group | Los Angeles | CA | 3320 S Hill St
South East LA
Los Angeles, CA 90007 | 34.01971 | -118.274281 | 3.5 | 2 | True | ['Medical Centers', 'Health and Medical'] | ['South East LA'] | ['University of Southern California'] | http://s3-media1.ak.yelpcdn.com/bphoto/SdUWxREuWuPvvot6faxfXg/ms.jpg | http://www.yelp.com/biz/southern-california-medical-group-los-angeles | business
ae5tm46-ewAhUJ6HR4fXFA | Harvard Square Shiatsu | Cambridge | MA | 12 Arrow St
Ste 108
Harvard Square
Cambridge, MA 02138 | 42.3711328 | -71.1148158 | 4.0 | 4 | True | ['Massage', 'Beauty and Spas'] | ['Harvard Square'] | ['Massachusetts Institute of Technology', 'Harvard University'] | http://s3-media1.ak.yelpcdn.com/bphoto/IGjhD4m7mexe7N06OX9DcA/ms.jpg | http://www.yelp.com/biz/harvard-square-shiat

In [ ]:
spark.sql("SELECT user_id, business_id FROM yelp.reviews LIMIT 3").display()
spark.sql("SELECT user_id FROM yelp.users LIMIT 3").display()
spark.sql("SELECT business_id FROM yelp.businesses LIMIT 3").display()

user_id | business_id
kT43SxDgMGzbeXpO51f0hQ | wbpbaWBfU54JbjLIDwERQA
T9hGHsbJW9Hw1cJAlIAWmw | 4iTRjN_uAdAb7_YZDVHJdg
Z_WAxc4RUpKp3y12BH1bEg | qw5gR8vW7mSOK4VROSwdMA
user_id
JkeCKyEaQlbLd9uZYl4DjA
cs91PAsv6esdWAaSkzm2lg
cMgGj2FXHEbzdNZdLN_EwA
business_id
Iu-oeVzv8ZgP18NIB0UMqg
ae5tm46-ewAhUJ6HR4fXFA
lb9qELwduFgBo408x2VKOA

In [ ]:
%sql
CREATE OR REPLACE TABLE yelp.reviews_silver AS
SELECT
  r.review_id,
  r.user_id,
  r.business_id,
  r.stars          AS review_stars,
  r.date           AS review_date,
  r.text           AS review_text,
  r.votes          AS review_votes,

  u.name           AS user_name,
  u.average_stars  AS user_average_stars,
  u.review_count   AS user_review_count,
  u.url            AS user_url,
  u.votes          AS user_votes,

  b.name           AS business_name,
  b.city,
  b.state,
  b.full_address,
  b.latitude,
  b.longitude,
  b.stars          AS business_stars,
  b.review_count   AS business_review_count,
  b.open,
  b.categories,
  b.neighborhoods,
  b.schools,
  b.photo_url,
  b.url            AS business_url

FROM yelp.reviews r
LEFT JOIN yelp.users u      ON CAST(r.user_id AS STRING)     = CAST(u.user_id AS STRING)
LEFT JOIN yelp.businesses b ON CAST(r.business_id AS STRING) = CAST(b.business_id AS STRING);

num_affected_rows | num_inserted_rows

In [ ]:
# Verify
spark.sql("SELECT * FROM yelp.reviews_silver LIMIT 10").display()
print("reviews_silver row count:", spark.sql("SELECT COUNT(*) FROM yelp.reviews_silver").collect()[0][0])
print("reviews_silver columns:", spark.table("yelp.reviews_silver").columns)

review_id | user_id | business_id | review_stars | review_date | review_text | review_votes | user_name | user_average_stars | user_review_count | user_url | user_votes | business_name | city | state | full_address | latitude | longitude | business_stars | business_review_count | open | categories | neighborhoods | schools | photo_url | business_url
0xuZfa0t4MNWd3eIFF02ug | kT43SxDgMGzbeXpO51f0hQ | wbpbaWBfU54JbjLIDwERQA | 5.0 | 2009-06-09 | I'm a fan of soft serve ice cream and Guptill's Coney Express has delicious ice cream with many flavors.  I've tried Kurver Kreme in Colonie, Tastee Freeze in Delmar and Country Drive Inn in Clifton Park, but I think that this place has the best soft serve ice cream.  The portions are generous and the taste is very rich.  For example, the brownie sundae is decadently delicious but likely too much for one person.  They also have cupcake sundaes which I am looking to try soon! | [1, 0, 0] | Caroline A. | 4.25 | 9 | http://www.yelp.com/user_details?us

## Part B. Analyze the wikipedia dataset.

The data for Part B is one of Databricks' sample datasets, with the following Unity Catalog path:

`/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed`

The data has 6 fields with the following meanings:

- `prev_id`: If the referer does not correspond to an article in the main namespace of English Wikipedia, this value will be empty. Otherwise, it contains the unique MediaWiki page ID of the article corresponding to the referer, i.e., the previous article the client was on.
- `curr_id`: The MediaWiki unique page ID of the article the client requested.
- `prev_title`: The result of mapping the referer URL to the fixed set of values described below, or `referer`.
- `curr_title`: The title of the article the client requested.
- `n`: The number of occurrences of the (referer, resource) pair, or `hits`.
- `type`:
  + "`link`" if the referer and request are both articles and the referer links to the request.
  + "`redlink`" if the referer is an article and links to the request, but the request is not in the production enwiki.page table.
  + "`other`" if the referer and request are both articles but the referer does not link to the request. This can happen when clients search or spoof their referer.

Referers (`prev_title`) were mapped to a fixed set of values corresponding to internal traffic or external traffic from one of the top 5 global traffic sources to English Wikipedia, based on this scheme:

 Referer Description                                                      | prev_title value    |
--------------------------------------------------------------------------|---------------------|
 an article in the main namespace of English Wikipedia                    | the article title   |
 any Wikipedia page that is not in the main namespace of English Wikipedia | other-wikipedia     |
 an empty referer                                                         | other-empty         |
 a page from any other Wikimedia project                                  | other-internal      |
 Google                                                                  | other-google        |
 Yahoo                                                                   | other-yahoo         |
 Bing                                                                    | other-bing          |
 Facebook                                                                | other-facebook      |
 Twitter                                                                 | other-twitter       |
 anything else                                                           | other-other         |

In the second line of the file, for instance, we can see there were 121 clicks from Google to the Wikipedia page on "!!" (double exclamation marks). People search for everything!

- prev_id = (nothing)
- curr_id = 3632887 --> (Wikipedia page ID)
- n = 121 (People clicked from Google to this page 121 times in this month.)
- prev_title = other-google (This data record is for referrals from Google.)
- curr_title = `!!` (This Wikipedia page is about a double exclamation mark.)
- type = other

### 8. Display the list of files in the folder (using `dbuils` APIs)



In [ ]:
display(dbutils.fs.ls("/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed"))

path | name | size | modificationTime
dbfs:/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed/2015_2_clickstream.tsv | 2015_2_clickstream.tsv | 1322171548 | 1596690797000

### 9. Display the first 1000 bytes in the TSV file 
- Use the API `dbutils.fs.head(path, n_bytes)`, where the second argument will control how many bytes to show.

In [ ]:
files = dbutils.fs.ls("/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed")
print(dbutils.fs.head(files[0].path, 1000))

[Truncated to first 1000 bytes]
prev_id	curr_id	n	prev_title	curr_title	type
	3632887	121	other-google	!!	other
	3632887	93	other-wikipedia	!!	other
	3632887	46	other-empty	!!	other
	3632887	10	other-other	!!	other
64486	3632887	11	!_(disambiguation)	!!	other
2061699	2556962	19	Louden_Up_Now	!!!_(album)	link
	2556962	25	other-empty	!!!_(album)	other
	2556962	16	other-google	!!!_(album)	other
	2556962	44	other-wikipedia	!!!_(album)	other
64486	2556962	15	!_(disambiguation)	!!!_(album)	link
600744	2556962	297	!!!	!!!_(album)	link
	6893310	11	other-empty	!Hero_(album)	other
1921683	6893310	26	!Hero	!Hero_(album)	link
	6893310	16	other-wikipedia	!Hero_(album)	other
	6893310	23	other-google	!Hero_(album)	other
8127304	22602473	16	Jericho_Rosales	!Oka_Tokat	link
35978874	22602473	20	List_of_telenovelas_of_ABS-CBN	!Oka_Tokat	link
	22602473	57	other-google	!Oka_Tokat	other
	22602473	12	other-wikipedia	!Oka_Tokat	other
	22602473	23	other-empty	!Oka_Tokat	other
7360687	22602473	10	Rica_Peralejo	

### 10. Load the data into a DataFrame `df`, providing a DDL schema
- When you specify a schema, Spark will not scan the data to infer the schema and thus can run faster.
- Set `prev_id` and `curr_id` to be of `long` type.
- Verify the resulting DataFrame by displaying the first 10 rows and the count of rows.

In [ ]:
schema = "prev_id LONG, curr_id LONG, prev_title STRING, curr_title STRING, n INT, type STRING"

df = spark.read \
    .option("sep", "\t") \
    .option("header", "false") \
    .schema(schema) \
    .csv("/databricks-datasets/wikipedia-datasets/data-001/clickstream/raw-uncompressed")

df.limit(10).display()
print("Total row count:", df.count())

prev_id | curr_id | prev_title | curr_title | n | type
None | None | n | prev_title | None | type
None | 3632887 | 121 | other-google | None | other
None | 3632887 | 93 | other-wikipedia | None | other
None | 3632887 | 46 | other-empty | None | other
None | 3632887 | 10 | other-other | None | other
64486 | 3632887 | 11 | !_(disambiguation) | None | other
2061699 | 2556962 | 19 | Louden_Up_Now | None | link
None | 2556962 | 25 | other-empty | None | other
None | 2556962 | 16 | other-google | None | other
None | 2556962 | 44 | other-wikipedia | None | other
Total row count: 22509898


### 11. Sample the data and use the sampled data to create a new table  

- The new table will be `clickstream` under the new database `wikipedia`.
- Sample 10% (0.1) with seed 1234.
- Verify the resulting table by displaying the count of rows and the first 10 rows.

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS wikipedia;

In [ ]:
df.sample(fraction=0.1, seed=1234) \
  .write.mode("overwrite") \
  .saveAsTable("wikipedia.clickstream")

# Verification
spark.sql("SELECT * FROM wikipedia.clickstream LIMIT 10").display()
print("Sampled row count:", spark.sql("SELECT COUNT(*) FROM wikipedia.clickstream").collect()[0][0])

prev_id | curr_id | prev_title | curr_title | n | type
None | 33287902 | 12 | other-wikipedia | None | other
None | 336446 | 141 | other-google | None | other
None | 336446 | 11 | other-bing | None | other
None | 17753292 | 47 | other-other | None | other
178771 | 17753292 | 25 | Starbucks | None | link
31188 | 29000 | 24 | Triassic–Jurassic_extinction_event | None | link
None | 29000 | 1059 | other-empty | None | other
219268 | 29000 | 20 | Population_genetics | None | link
26569537 | 29000 | 47 | Recent_African_origin_of_modern_humans | None | other
44862806 | 29000 | 10 | Outline_of_evolution | None | link
Sampled row count: 2252530


### 12. What are the top 10 articles by the number of hits within the month?

> Each entry in the table has an `n` field capturing the number of hits on the article `curr_title` *from the source `prev_title`* in this month. The total hits for the article would be the sum of hits from all sources.

- Please note that in the output, the column titles should be `page_title` and `total_hits`.

In [ ]:
%sql
SELECT 
  curr_title  AS page_title,
  SUM(n)      AS total_hits
FROM wikipedia.clickstream
GROUP BY curr_title
ORDER BY total_hits DESC
LIMIT 10;

page_title | total_hits
Fate:_Undiscovered_Realms | 2147483647
Mersenne_prime | 2147483647
Boom_Boom_Satellites | 19972007
Future_Imperfect | 11001001
A_Crazy_Steal | 5317704
Megatone | 4092001
Venetian_Snares | 2370894
Floor_Jansen | 1011001
Daniel_Gildenlöw | 1011001
Elected | 1011001

### 13. Which referers sent the most traffic to Wikipedia in Feb 2015?

- The output should use `referer` (for `prev_title`) and `tot_hits` as column titles.
- Limit your results to the top 10 referers.
- Create a vertical bar chart with the most frequent referer displayed on the left.

In [ ]:
%sql
SELECT 
  prev_title  AS referer,
  SUM(n)      AS tot_hits
FROM wikipedia.clickstream
GROUP BY prev_title
ORDER BY tot_hits DESC
LIMIT 10;

referer | tot_hits
12 | 2147635888
49 | 2147499806
24 | 20008212
30 | 11103839
119 | 5319174
18 | 4135085
55 | 2381802
36 | 1029393
33 | 1022142
90 | 1017894

### 14. (Challenging) What are the incoming and outgoing hits for each page?

An analyst wants to gain insights into page performance, including their incoming hits, outgoing hits, and the ratio between the two.

- Save the results in a new table `pages` in the `wikipedia` database.
- The table should have `page_id`, `page` (i.e., the title of the page), `in_count` (hits where the page is the `curr_title`), `out_count` (hits when the page is the `prev_title`), and `out_in_ratio` (out_count/in_count).
- Please note that not all pages have both incoming and outgoing hits. Some only have incoming hits, others may only have outgoing hits. In your results, you need to include all cases.
- After you create the table, verify it by displaying the first 10 rows and the count.

> TIP: 
> - `in_count` and `out_count` need to computed separately, and then need to be joined. You can do this with CTE (common table expression) or step by step. 

In [ ]:
%sql
CREATE OR REPLACE TABLE wikipedia.pages AS
WITH incoming AS (
  SELECT
    curr_id       AS page_id,
    curr_title    AS page,
    SUM(n)        AS in_count
  FROM wikipedia.clickstream
  GROUP BY curr_id, curr_title
),
outgoing AS (
  SELECT
    prev_id       AS page_id,
    prev_title    AS page,
    SUM(n)        AS out_count
  FROM wikipedia.clickstream
  WHERE prev_id IS NOT NULL
  GROUP BY prev_id, prev_title
)
SELECT
  COALESCE(i.page_id, o.page_id)                  AS page_id,
  COALESCE(i.page, o.page)                         AS page,
  i.in_count,
  o.out_count,
  ROUND(o.out_count / NULLIF(i.in_count, 0), 4)   AS out_in_ratio
FROM incoming i
FULL OUTER JOIN outgoing o
  ON i.page_id = o.page_id AND i.page = o.page;

num_affected_rows | num_inserted_rows

In [ ]:
# Verification
spark.sql("SELECT * FROM wikipedia.pages LIMIT 10").display()
print("Pages count:", spark.sql("SELECT COUNT(*) FROM wikipedia.pages").collect()[0][0])

page_id | page | in_count | out_count | out_in_ratio
None | Rowthiram | None | None | None
43417950 | Flemming_Muus | None | None | None
14131347 | Yohimbine | None | None | None
512487 | other-google | None | None | None
57992 | Rustproofing | None | None | None
418883 | other-google | None | None | None
12618545 | other-wikipedia | None | None | None
3754390 | Local_nonsatiation | None | None | None
10574673 | other-google | None | None | None
1266447 | other-bing | None | None | None
Pages count: 3603591


### 15. Export `pages` in CSV format

- Export to the `homework2` volume you created earlier, placing the exported files in the `pages` folder.
- The exported CSV file(s) should include a header row.
- After exporting to CSV, verify by displaying the files in the folder and the first 1000 characters of one CSV file.

In [ ]:
path = "/Volumes/msbabigdata/spark/homework2/pages"

spark.table("wikipedia.pages") \
    .write.mode("overwrite") \
    .option("header", "true") \
    .csv(path)

# Verification
display(dbutils.fs.ls(path))

#first 1000 chars of the CSV
csv_file = [f.path for f in dbutils.fs.ls(path) if f.name.endswith(".csv")][0]
print(dbutils.fs.head(csv_file, 1000))

path | name | size | modificationTime
dbfs:/Volumes/msbabigdata/spark/homework2/pages/_SUCCESS | _SUCCESS | 0 | 1772579805000
dbfs:/Volumes/msbabigdata/spark/homework2/pages/_committed_8281981678382788261 | _committed_8281981678382788261 | 113 | 1772579805000
dbfs:/Volumes/msbabigdata/spark/homework2/pages/_started_8281981678382788261 | _started_8281981678382788261 | 0 | 1772579802000
dbfs:/Volumes/msbabigdata/spark/homework2/pages/part-00000-tid-8281981678382788261-6a0ddeef-e3d6-4320-a717-1b6ca2c3b938-281-1-c000.csv | part-00000-tid-8281981678382788261-6a0ddeef-e3d6-4320-a717-1b6ca2c3b938-281-1-c000.csv | 83278552 | 1772579802000
[Truncated to first 1000 bytes]
page_id,page,in_count,out_count,out_in_ratio
,Rowthiram,,,
43417950,Flemming_Muus,,,
14131347,Yohimbine,,,
512487,other-google,,,
57992,Rustproofing,,,
418883,other-google,,,
12618545,other-wikipedia,,,
3754390,Local_nonsatiation,,,
10574673,other-google,,,
1266447,other-bing,,,
38933,Paraclete,,,
34545425,Monte_Carlo,,,
272657

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.homework2;
  

## Clean Up
As part of an idempotent workflow, we want to clean up the tables, volumes, files, and databases created in this homework.

### 16. Write your clean up code below

In [ ]:
%sql

drop database if exists yelp cascade;
drop database if exists wikipedia cascade;
drop volume if exists msbabigdata.spark.homework2;

This is the end of the homework.